In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**VGG19 training**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split, Subset
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import os
import random

#Dataset path
data_dir = "/content/drive/MyDrive/thesis/data/asl_alphabet_train/asl_alphabet_train"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(data_dir, transform=transform)
num_classes = len(full_dataset.classes)
print(f"Total images in dataset: {len(full_dataset)} | Classes: {num_classes}")

indices = []

for class_idx in range(num_classes):
    class_indices = [i for i, (_, label) in enumerate(full_dataset.samples) if label == class_idx]
    half_class_size = len(class_indices) // 2
    indices.extend(class_indices[:half_class_size])

half_dataset = Subset(full_dataset, indices)
print(f"Using {len(half_dataset)} images for this experiment (first 50% from each class).")

#Splitting dataset into 80-20 training-validation set
train_size = int(0.8 * len(half_dataset))
val_size = len(half_dataset) - train_size
train_dataset, val_dataset = random_split(half_dataset, [train_size, val_size])
print(f"Training images: {len(train_dataset)} | Validation images: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

#Loading pretrained VGG19
vgg19 = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
for param in vgg19.features.parameters():
    param.requires_grad = False  # freeze convolution layers

vgg19.classifier[6] = nn.Linear(4096, num_classes)
vgg19 = vgg19.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg19.classifier[6].parameters(), lr=1e-4)
scaler = GradScaler()

epochs = 3
for epoch in range(epochs):
    vgg19.train()
    train_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast():
            outputs = vgg19(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}%")

#Validation
vgg19.eval()
val_correct, val_total = 0, 0
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Validating"):
        images, labels = images.to(device), labels.to(device)
        outputs = vgg19(images)
        _, predicted = torch.max(outputs, 1)
        val_total += labels.size(0)
        val_correct += (predicted == labels).sum().item()

val_acc = 100 * val_correct / val_total
print(f"Validation Accuracy: {val_acc:.2f}%")

#Saving trained model
save_path = "/content/drive/MyDrive/thesis/trained_vgg19_half_split.pth"
torch.save(vgg19.state_dict(), save_path)
print(f"Model saved to {save_path}")

**Testing with extended modified test set**

In [ ]:
!rm -rf /content/test_set_8596_hand_flipped
!cp -r "/content/drive/MyDrive/thesis/large_asl_test_set/test_set_8596_hand_flipped" "/content/test_set_8596_hand_flipped"
!find /content/test_set_8596_hand_flipped -type f \( -iname "*.jpg" -o -iname "*.png" -o -iname "*.jpeg" \) | wc -l

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

torch.backends.cudnn.benchmark = True

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/thesis/trained_vgg19_half_split.pth"
TEST_DIR   = "/content/test_set_8596_hand_flipped"

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder(TEST_DIR, transform=transform)
print("Test classes found:", test_dataset.classes)
print("Num test images:", len(test_dataset))

test_loader = DataLoader(
    test_dataset,
    batch_size=128,          # faster
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [ ]:
train_class_names = [
    'A','B','C','D','E','F','G','H','I','J',
    'K','L','M','N','O','P','Q','R','S','T',
    'U','V','W','X','Y','Z','del','nothing','space'
]

vgg19 = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
for p in vgg19.features.parameters():
    p.requires_grad = False

vgg19.classifier[6] = nn.Linear(4096, len(train_class_names))

vgg19.load_state_dict(torch.load(MODEL_PATH, map_location=device))
vgg19 = vgg19.to(device)
vgg19.eval()

print("VGG19 checkpoint:", MODEL_PATH)


In [ ]:
test_class_names = test_dataset.classes
per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)

correct = 0
total = 0

y_true_names = []
y_pred_names = []

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = vgg19(images)
        preds = outputs.argmax(dim=1)

        pred_names = [train_class_names[i] for i in preds.cpu().numpy()]
        true_names = [test_class_names[i] for i in labels.cpu().numpy()]

        for tn, pn in zip(true_names, pred_names):
            total += 1
            per_class_total[tn] += 1
            y_true_names.append(tn)
            y_pred_names.append(pn)

            if tn == pn:
                correct += 1
                per_class_correct[tn] += 1

overall_acc = 100 * correct / total
print(f"\n Overall Accuracy: {overall_acc:.2f}% ({correct}/{total})\n")

print("Per-class accuracy:")
for c in test_class_names:
    acc = 100 * per_class_correct[c] / per_class_total[c] if per_class_total[c] else 0.0
    print(f"{c:8}: {acc:6.2f}% ({per_class_correct[c]}/{per_class_total[c]})")

**Confusion matrix for alphabet level analysis

In [ ]:
test_name_to_idx = {name: i for i, name in enumerate(test_class_names)}

filtered_true = []
filtered_pred = []
dropped = 0

for tn, pn in zip(y_true_names, y_pred_names):
    if pn in test_name_to_idx:
        filtered_true.append(test_name_to_idx[tn])
        filtered_pred.append(test_name_to_idx[pn])
    else:
        dropped += 1

print(f"\nPredictions outside test label set dropped from confusion matrix: {dropped}")

cm = confusion_matrix(filtered_true, filtered_pred, labels=list(range(len(test_class_names))))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm = np.nan_to_num(cm_norm)

plt.figure(figsize=(7, 7))
plt.imshow(cm_norm, interpolation="nearest")
plt.title("Confusion Matrix (Normalized) – VGG19 (Test Classes)")
plt.colorbar()
plt.xticks(range(len(test_class_names)), test_class_names, rotation=90)
plt.yticks(range(len(test_class_names)), test_class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()
